In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import chi2
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# **Task 1**

In [2]:
n = 500   # number of observations
p = 50    # number of all features
k = 10    # number of significant features

In [3]:
# dataset 1
def generate_dataset1(n=500, p=50, k=10, random_state=None):
    rng = np.random.default_rng(random_state)

    X = rng.normal(0, 1, size=(n, p))

    threshold = chi2.ppf(0.5, df=k)

    signal = np.sum(X[:, :k] ** 2, axis=1)

    y = (signal > threshold).astype(int)

    return X, y

In [4]:
# dataset 2
def generate_dataset2(n=500, p=50, k=10, random_state=None):
    rng = np.random.default_rng(random_state)

    X = rng.normal(0, 1, size=(n, p))

    signal = np.sum(np.abs(X[:, :k]), axis=1)

    y = (signal > k).astype(int)

    return X, y

In [5]:
X, y = generate_dataset1(n=500, p=50, k=10, random_state=42)

print(X.shape)
print(y.shape)
print(np.mean(y))

(500, 50)
(500,)
0.474


**Are the relevant variables, X1,X2,...,Xk, expected to be detectable by simple marginal correlation with Y? Why or why not?**

No, the relevant variables are not expected to be easily detectable by simple marginal correlation with **Y**. In both datasets, the relationship between each relevant feature and the response is non-linear and symmetric. For example, in Dataset 1 the response depends on **X_j^2**, so both large positive and large negative values of **X_j** are associated with **Y = 1**. Because the feature distribution is symmetric around zero, the linear correlation between **X_j** and **Y** can be close to zero. The same idea applies to Dataset 2, where the response depends on **|X_j|**. Therefore, marginal correlation may fail even though the variables are truly important.


# **Task 2**

- Random Forest - mean descrease impurity
- Random Forest - permutation importance
- Boruta

In [6]:
# random forest impurity

X, y = generate_dataset1(n=500, p=50, k=10, random_state=42)

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf.fit(X, y)

mdi_importance = rf.feature_importances_

In [7]:
ranking_mdi = np.argsort(mdi_importance)[::-1]

print("Top 10 features according to MDI:")
print(ranking_mdi[:10])

Top 10 features according to MDI:
[3 5 4 2 6 0 8 7 9 1]


In [8]:
true_features = set(range(k))
top_k_mdi = set(ranking_mdi[:k])

print("True features:", true_features)
print("Top k MDI features:", top_k_mdi)
print("Successfully recovered all relevant features:", true_features == top_k_mdi)

True features: {0, 1, 2, 3, 4, 5, 6, 7, 8, 9}
Top k MDI features: {np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)}
Successfully recovered all relevant features: True


In [9]:
# permutation importance
perm = permutation_importance(
    rf,
    X,
    y,
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

perm_importance = perm.importances_mean
ranking_perm = np.argsort(perm_importance)[::-1]

print("Top 10 features according to permutation importance:")
print(ranking_perm[:10])

Top 10 features according to permutation importance:
[ 3  5  2  9  8 45 47 46 41 40]


In [10]:
# sprawdzenie
top_k_perm = set(ranking_perm[:k])

print("Top k permutation features:", top_k_perm)
print("Successfully recovered all relevant features:", true_features == top_k_perm)

Top k permutation features: {np.int64(2), np.int64(3), np.int64(5), np.int64(8), np.int64(9), np.int64(41), np.int64(40), np.int64(45), np.int64(46), np.int64(47)}
Successfully recovered all relevant features: False


In [14]:
# boruta
# !pip install boruta

from boruta import BorutaPy

rf_boruta = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    max_depth=None
)

boruta = BorutaPy(
    estimator=rf_boruta,
    n_estimators='auto',
    random_state=42,
    verbose=0
)

boruta.fit(X, y)

selected_boruta = np.where(boruta.support_)[0]

print("Selected features by Boruta:")
print(selected_boruta)

Selected features by Boruta:
[0 1 2 3 4 5 6 7 8 9]


In [15]:
# sprawdzenie
true_features = set(range(k))
selected_boruta_set = set(selected_boruta)

print("True features:", true_features)
print("Boruta selected:", selected_boruta_set)
print("All relevant features selected:", true_features.issubset(selected_boruta_set))

True features: {0, 1, 2, 3, 4, 5, 6, 7, 8, 9}
Boruta selected: {np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)}
All relevant features selected: True


#### **(b) Try different values of n, p and k. First, you can use values: n = 500, p = 50 and k= 10.**

In [16]:
settings = [
    (500, 50, 10),
    (200, 50, 10),
    (500, 100, 10),
    (500, 50, 5),
    (1000, 50, 10)
]

In [17]:
for n, p, k in settings:
    X, y = generate_dataset1(n=n, p=p, k=k, random_state=42)

    rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
    rf.fit(X, y)

    mdi_importance = rf.feature_importances_
    ranking_mdi = np.argsort(mdi_importance)[::-1]

    top_k = set(ranking_mdi[:k])
    true_features = set(range(k))

    print(f"n={n}, p={p}, k={k}")
    print("Success:", top_k == true_features)
    print("Top k:", sorted(top_k))
    print()

n=500, p=50, k=10
Success: True
Top k: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]

n=200, p=50, k=10
Success: False
Top k: [np.int64(0), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(9), np.int64(34), np.int64(43)]

n=500, p=100, k=10
Success: True
Top k: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]

n=500, p=50, k=5
Success: True
Top k: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

n=1000, p=50, k=10
Success: True
Top k: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]



#### **(c) Repeat data generation process L= 50 times**

In [18]:
def check_recovery_mdi(dataset_function, n=500, p=50, k=10, seed=0):
    X, y = dataset_function(n=n, p=p, k=k, random_state=seed)

    rf = RandomForestClassifier(
        n_estimators=300,
        random_state=seed,
        n_jobs=-1
    )
    rf.fit(X, y)

    importance = rf.feature_importances_
    ranking = np.argsort(importance)[::-1]

    top_k = set(ranking[:k])
    true_features = set(range(k))

    return top_k == true_features

In [ ]:
L = 50

successes = []

for seed in range(L):
    success = check_recovery_mdi(
        generate_dataset1,
        n=500,
        p=50,
        k=10,
        seed=seed
    )
    successes.append(success)

prob_success = np.mean(successes)

print("Probability of successful feature recovery:")
print(prob_success)